[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/38_grpo_loss_solution.ipynb)

# 参考解答：组相对策略优化损失（GRPO）

DeepSeekMath／DeepSeek-R1 标准词元级 GRPO 损失参考解答。

In [ ]:
# 在 Colab 中安装 torch-judge（JupyterLab/Docker 中会跳过）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor

In [ ]:
# ✅ 参考解答

def grpo_loss(new_logps: Tensor,
              old_logps: Tensor,
              rewards: Tensor,
              group_ids: Tensor,
              mask: Tensor | None = None,
              ref_logps: Tensor | None = None,
              clip_ratio: float = 0.2,
              kl_coef: float = 0.0,
              eps: float = 1e-6) -> Tensor:
    """标准词元级组相对策略优化（GRPO）损失。"""
    r_detached = rewards.detach().to(dtype=new_logps.dtype, device=new_logps.device)
    advantages = torch.zeros_like(r_detached)

    for gid in group_ids.unique():
        g_mask = group_ids == gid
        r_g = r_detached[g_mask]
        mean_g = r_g.mean()
        std_g = r_g.std(unbiased=False)
        advantages[g_mask] = (r_g - mean_g) / (std_g + eps)

    if mask is None:
        token_mask = torch.ones_like(new_logps)
    else:
        token_mask = mask.to(dtype=new_logps.dtype, device=new_logps.device)

    log_diff = torch.where(
        token_mask > 0,
        new_logps - old_logps.detach(),
        torch.zeros_like(new_logps),
    )
    ratios = torch.exp(log_diff)
    adv_2d = advantages.detach().unsqueeze(-1)

    unclipped = ratios * adv_2d
    clipped = torch.clamp(ratios, 1.0 - clip_ratio, 1.0 + clip_ratio) * adv_2d
    per_token_obj = torch.min(unclipped, clipped)

    if ref_logps is not None and kl_coef > 0.0:
        delta = torch.where(
            token_mask > 0,
            ref_logps.detach() - new_logps,
            torch.zeros_like(new_logps),
        )
        k3_kl = torch.exp(delta) - delta - 1.0
        per_token_obj = per_token_obj - kl_coef * k3_kl

    seq_lengths = token_mask.sum(dim=-1).clamp_min(1.0)
    per_seq_loss = -(per_token_obj * token_mask).sum(dim=-1) / seq_lengths
    return per_seq_loss.mean()

In [ ]:
# 示例
ratios = torch.tensor([[1.30, 1.10], [0.70, 0.90]])
old_logps = torch.zeros(2, 2)
new_logps = ratios.log().requires_grad_(True)
rewards = torch.tensor([1.0, -1.0])
group_ids = torch.tensor([0, 0])
print('Loss:', grpo_loss(new_logps, old_logps, rewards, group_ids, eps=0.0).item())

In [ ]:
from torch_judge import check
check('grpo_loss')